# 4_1_2 - Analyses statistiques thématiques

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv(
    "../data/interim/df_repu.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [3]:
df_thématique = pd.read_csv(
    "../data/interim/education.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [4]:
print("shape avant fusion:", df.shape)

# Merge et virer la col id pour éviter doublon
df = df.merge(
    df_thématique,
    left_on="ID_paragraphe",
    right_on="dataset_ID_paragraphe",
    how="left",
).drop(columns=["dataset_ID_paragraphe"])  # supprimer les colonnes non-utiles de la prédiction pour ne garder que "autre"/"thématique"

print("shape après fusion:", df.shape)


shape avant fusion: (13729, 53)
shape après fusion: (13745, 58)


In [5]:
df.drop(columns=["id", "Autre", "Education", "entropy"])  # supprimer les colonnes non-utiles de la prédiction pour ne garder que "autre"/"thématique"

,UID,SeanceRef,SessionRef,DateSeance,DateSeanceJour,NumSeanceJour,NumSeance,TypeAssemblee,Legislature,Session,...,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,active,dateMaj,DateSeance_ts,parti_affiliation,repu_match_valide,prediction
0,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,0.19,0.11,0.970,0.923,0.0,2025-09-26,2018-06-02 09:30:00,MODEM,True,Autre
1,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,0.48,0.72,0.897,0.323,0.0,2025-09-26,2018-06-02 09:30:00,LR,True,Autre
2,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,NaN,NaN,NaN,NaN,NaN,NaN,2018-06-02 09:30:00,GVT,True,Autre
3,CRSANR5L15S2019O1N178,NaN,NaN,20190314150000000,jeudi 14 mars 2019,2,178,AN,15,Session ordinaire 2018-2019,...,0.16,0.24,0.945,0.000,1.0,2025-09-26,2019-03-14 15:00:00,PCF,True,Autre
4,CRSANR5L15S2018O1N279,NaN,NaN,20180619150000000,mardi 19 juin 2018,1,279,AN,15,Session ordinaire 2017-2018,...,0.24,0.53,0.992,0.992,0.0,2025-09-26,2018-06-19 15:00:00,REN,True,Autre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13740,CRSANR5L16S2024O1N132,RUANR5L16S2024IDS28086,SCR5A2024O1,20240229150000000,jeudi 29 février 2024,2,132,AN,16,Session ordinaire 2023-2024,...,0.24,0.43,0.953,0.000,1.0,2025-09-26,2024-02-29 15:00:00,PS,True,Autre
13741,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,0.05,0.00,0.969,0.000,1.0,2025-09-26,2023-02-10 15:00:00,LR,True,Autre
13742,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,0.35,0.57,0.948,0.948,0.0,2025-09-26,2023-02-10 15:00:00,REN,True,Autre
13743,CRSANR5L16S2023O1N142,RUANR5L16S2023IDS26788,SCR5A2023O1,20230210150000000,vendredi 10 février 2023,2,142,AN,16,Session ordinaire 2022-2023,...,0.10,0.27,0.926,0.000,1.0,2025-09-26,2023-02-10 15:00:00,REN,True,Autre


In [6]:
# Filtrer le DF pour n'avoir que les utilisations de la thématique escomptée 
df = df[df["prediction"] == "Education"]
df.shape

(1658, 58)

In [23]:
import csv
df.to_csv(
    "../data/interim/data_AT_educ.csv",
    index=False,
    quoting=csv.QUOTE_ALL,  # a permis de résoudre le soucis d'écart. Checker
)

In [7]:
import datetime
import locale

# Active la locale française
locale.setlocale(locale.LC_TIME, "fr_FR.UTF-8")

'fr_FR.UTF-8'

In [8]:
df["DateSeance_ts"] = pd.to_datetime(df["DateSeanceJour"], format="%A %d %B %Y")
df["DateSeance_day"] = df["DateSeance_ts"].dt.normalize()  

## Analyses générales 

### Par jours

#### Sur le corpus intégral

In [12]:
# afficher les 25 dates les plus fréquentes sous forme de tableau
table = df["DateSeance_day"].value_counts()[0:20].reset_index()
table = table.rename(columns={"count": "Nombre de mentions"})
table

,DateSeance_day,Nombre de mentions
0,2021-02-11,120
1,2019-02-11,58
2,2019-02-12,40
3,2021-02-12,39
4,2024-01-17,38
5,2019-02-15,35
6,2023-01-12,34
7,2023-01-10,32
8,2019-02-13,31
9,2018-03-28,29


**Remarques**

- 2 du top 10 sont pendant la discussion de la loi séparatisme (11 et 12 février 2021) dont 120 le 11 février.
- 11, 12, 13 et 15 février 2019 (pendant la discussion sur l'école de la confiance).
- 17 janvier 2024 : Instrumentalisation politique des élections des parents d’élèves dans les conseils d’école.
- 10 et 12 janvier 2023 : sur le Port d’une tenue uniforme à l’école
- 28 mars 2018 lors d'une discussion sur les Écoles privées hors contrat



In [14]:
# Graphique par jour (illisible)
fig_time = px.bar(df.resample("D", on="DateSeance_day").size())

fig_time.update_layout(
    title="Nombre de mentions de la notion de république",  # ajouter un titre
    xaxis_title="Date",
    yaxis_title="Nombre de mentions",  # renommer les étiquettes d'axes
    template="plotly_white",  # changer le style du graphique
    showlegend=False,
)  # masquer la légende

# Afficher le graphique
fig_time.show()

#### Par jour sur corpus annualisé

In [15]:
# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["DateSeance_day"].dt.year == annee]

# aficher les 25 dates les plus fréquentes d'une année sous forme de tableau
table = (
    df_annee.resample("D", on="DateSeance_day")
      .size()
      .sort_values(ascending=False)
      .head(20)
      .reset_index()
      .rename(columns={0: "Nombre de mentions", "DateSeance_day": "Semaine"})
)
table

,Semaine,Nombre de mentions
0,2021-02-11,120
1,2021-02-12,39
2,2021-02-03,26
3,2021-04-08,26
4,2021-07-01,19
5,2021-06-30,18
6,2021-02-01,18
7,2021-06-28,18
8,2021-02-04,18
9,2021-12-01,17


In [16]:
# Graphique par jour sur une année 

# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["DateSeance_day"].dt.year == annee]

# Resampler par jour uniquement pour cette année
df_daily = df_annee.resample("D", on="DateSeance_day").size().reset_index()

# Tracer le graphique
fig_daily = px.bar(
    df_daily,
    x="DateSeance_day",
    y=0,
    title=f"Occurrences de l'idée de République en {annee} (par semaine)",
    labels={"DateSeance_day": "Date", "0": "Occurrences"},
    template="plotly_white",
)

fig_daily.update_layout(showlegend=False)

fig_daily.show()

### Par semaine

#### Par semaine sur corpus intégral

In [17]:
# aficher les 25 semaines les plus fréquentes sous forme de tableau
table = (
    df.resample("W", on="DateSeance_day")
      .size()
      .sort_values(ascending=False)
      .head(20)
      .reset_index()
      .rename(columns={0: "Nombre de mentions", "DateSeance_day": "Semaine"})
)

table

,Semaine,Nombre de mentions
0,2019-02-17,182
1,2021-02-14,166
2,2021-02-07,76
3,2023-01-15,68
4,2021-07-04,60
5,2018-04-01,55
6,2024-01-21,42
7,2021-04-11,29
8,2023-04-09,27
9,2020-10-25,26


**Remarques**


In [19]:
# Graphique par semaine
fig_time = px.bar(df.resample("W", on="DateSeance_day").size())

fig_time.update_layout(
    title="Nombre de mentions de la notion de république sur les thématiques éducatives",  # ajouter un titre
    xaxis_title="Date",
    yaxis_title="Nombre de mentions",  # renommer les étiquettes d'axes
    template="plotly_white",  # changer le style du graphique
    showlegend=False,
)  # masquer la légende

# Afficher le graphique
fig_time.show()


#### Par semaine sur corpus annualisé 

In [ ]:
# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["DateSeance_day"].dt.year == annee]

# aficher les 25 semaines les plus fréquentes d'une année sous forme de tableau
table = (
    df_annee.resample("W", on="DateSeance_day")
      .size()
      .sort_values(ascending=False)
      .head(20)
      .reset_index()
      .rename(columns={0: "Nombre de mentions", "DateSeance_day": "Semaine"})
)
table

,Semaine,Nombre de mentions
0,2021-02-07,625
1,2021-02-14,381
2,2021-07-04,269
3,2021-03-14,89
4,2021-10-31,82
5,2021-07-25,67
6,2021-05-09,63
7,2021-02-21,61
8,2021-12-05,58
9,2021-12-12,57


In [ ]:
# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["DateSeance_day"].dt.year == annee]

# Resampler par semaine uniquement pour cette année
df_weekly = df_annee.resample("W", on="DateSeance_day").size().reset_index()

# Tracer le graphique
fig_time = px.bar(
    df_weekly,
    x="DateSeance_day",
    y=0,
    title=f"Occurrences de l'idée de République en {annee} (par semaine)",
    labels={"DateSeance_day": "Date", "0": "Occurrences"},
    template="plotly_white",
)

fig_time.update_layout(showlegend=False)

fig_time.show()


### Par mois

#### Sur corpus intégral, par mois

In [21]:
# aficher les 10 mois les plus fréquentes sous forme de tableau
table = (
    df.resample("M", on="DateSeance_day")
      .size()
      .sort_values(ascending=False)
      .head(50)
      .reset_index()
      .rename(columns={0: "Nombre de mentions", "DateSeance_day": "Mois"})
)

table

# Rajouter les stats en moyenne et médiannes

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_2399/2819376984.py:3: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



,Mois,Nombre de mentions
0,2021-02-28,250
1,2019-02-28,213
2,2023-01-31,70
3,2018-03-31,68
4,2024-01-31,56
5,2020-10-31,53
6,2021-06-30,51
7,2024-05-31,46
8,2021-04-30,33
9,2021-12-31,33


In [23]:
fig_time = px.bar(df.resample("M", on="DateSeance_day").size())

fig_time.update_layout(
    title="Occurrence de l'idée de République (famille de mot)",  # ajouter un titre
    xaxis_title="Date",
    yaxis_title="Occurrences",  # renommer les étiquettes d'axes
    template="plotly_white",  # changer le style du graphique
    showlegend=False,
)  # masquer la légende

# Afficher le graphique
fig_time.show()

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_2399/2888636519.py:1: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



#### Par mois sur corpus annualisé

In [ ]:
# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["DateSeance_day"].dt.year == annee]

# aficher les 25 semaines les plus fréquentes d'une année sous forme de tableau
table = (
    df_annee.resample("M", on="DateSeance_day")
      .size()
      .sort_values(ascending=False)
      .head(20)
      .reset_index()
      .rename(columns={0: "Nombre de mentions", "DateSeance_day": "Semaine"})
)
table

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_1971/2613634599.py:9: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



,Semaine,Nombre de mentions
0,2021-02-28,1067
1,2021-06-30,348
2,2021-03-31,247
3,2021-05-31,174
4,2021-11-30,168
5,2021-10-31,153
6,2021-07-31,143
7,2021-12-31,137
8,2021-04-30,118
9,2021-01-31,80


In [ ]:
# Attention problème en x d'un décalage systématique de 1

# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["DateSeance_day"].dt.year == annee]

# Resampler par semaine uniquement pour cette année
df_monthly = df_annee.resample("M", on="DateSeance_day").size().reset_index()

# Tracer le graphique
fig_time = px.bar(
    df_monthly,
    x="DateSeance_day",
    y=0,
    title=f"Occurrences de l'idée de République en {annee} (par mois)",
    labels={"DateSeance_day": "Date", "0": "Occurrences"},
    template="plotly_white",
)

fig_time.update_layout(showlegend=False)

fig_time.show()

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_1971/319794865.py:8: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



#### Par an 

In [25]:
# Regrouper les occurrences par année
df_yearly = df.resample("Y", on="DateSeance_day").size().reset_index()
df_yearly["dateSeance_day"] = df_yearly["DateSeance_day"].dt.year  # garder juste l'année

# Tracer le graphique
fig_time = px.bar(
    df_yearly,
    x="DateSeance_day",
    y=0,
    title="Occurrence de l'idée de République (par année)",
    labels={"DateSeance_day": "Année", "0": "Occurrences"},
    template="plotly_white",
)

fig_time.update_layout(showlegend=False)  # masquer la légende

fig_time.show()


/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_2399/4139761760.py:2: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



## Par groupes 

In [26]:
df["parti_affiliation"].value_counts()

parti_affiliation
GVT       460
REN       268
LR        197
LFI       191
PS         97
PCF        92
MODEM      82
RN         73
UDI        55
LIOT       50
ECO        38
AGIR-E     22
HOR        18
NI         13
EDS         2
Name: count, dtype: int64

In [27]:
fig_partis_1 = px.bar(
    df["parti_affiliation"].value_counts()[0:15],
    # x=top_counts.index,
    # y=top_counts.values,
    labels={"value": "Nombre d'occurence", "Partis": "partis"},
    title="Occurrences de la 'République' en valeur absolue par 'partis' (+ Gouvernement)",
    template="plotly_white",
)
fig_partis_1.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
)
fig_partis_1.show()

In [84]:
df = df[df["parti_affiliation"] !="GVT"]
df.shape

(1198, 59)

In [85]:
def groupes_evolutions(df, date_col="DateSeance_day", parti_col="parti_affiliation", top_n=7):

    # Filtrer le top des partis
    top_partis = df[parti_col].value_counts().nlargest(top_n).index.tolist()

    # Filtrer les données pour ces partis uniquement
    df_top = df[df[parti_col].isin(top_partis)]

    # Grouper par année et parti
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="Y"), parti_col])
        .size()
        .reset_index(name="mentions")
    )

    # Extraire l'année pour affichage clair
    df_grouped["Année"] = df_grouped[date_col].dt.year

    # Tracer avec plotly express
    fig = px.line(
        df_grouped,
        x="Année",
        y="mentions",
        color=parti_col,
        markers=True,
        title=f"Évolution annuelle des appropriations scolaires de la République du top {top_n} partis",
        labels={"mentions": "Occurrences", parti_col: "Parti"}
    )

    fig.update_layout(
        xaxis=dict(dtick=1),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Parti"
    )

    fig.show()


In [86]:
groupes_evolutions(df)

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_2399/3560143010.py:11: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



In [47]:
# Top 10 partis
top_partis = df["parti_affiliation"].value_counts().index[:5].tolist()

# Grouper par ans
df_partis = df[df["parti_affiliation"].isin(top_partis)]
df_partis = (
    df_partis.groupby([pd.Grouper(key="DateSeance_day", freq="Y"), "parti_affiliation"])
    .size()
    .reset_index(name="mentions")
)

# Graphique
fig = px.bar(
        df_partis,
        x="DateSeance_day",
        y="parti_affiliation",
        title="titre",
        template="plotly_white",
    )
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()


/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_2399/3091162892.py:7: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



In [42]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Top 10 partis
top_partis = df["parti_affiliation"].value_counts().index[:5].tolist()

# Grouper par ans
df_partis = df[df["parti_affiliation"].isin(top_partis)]
df_partis = (
    df_partis.groupby([pd.Grouper(key="DateSeance_day", freq="Y"), "parti_affiliation"])
    .size()
    .reset_index(name="mentions")
)

cols = 4
rows = (len(top_partis) + cols - 1) // cols
max_y = df_partis["mentions"].max()

fig_orateurs_time = make_subplots(
    rows=rows, cols=cols, shared_xaxes=True, subplot_titles=top_partis
)

for idx, orateur in enumerate(top_partis):
    row = idx // cols + 1
    col = idx % cols + 1
    data_orateur = df_partis[df_partis["parti_affiliation"] == orateur]
    fig_orateurs_time.add_trace(
        go.Bar(
            x=data_orateur["DateSeance_day"], y=data_orateur["mentions"], name=orateur
        ),
        row=row,
        col=col,
    )

fig_orateurs_time.update_layout(
    height=300 * rows,
    width=1200,
    title_text="Dynamique temporelle des mentions par orateur",
    showlegend=False,
    template="plotly_white",
)

for row in range(1, rows + 1):
    for col in range(1, cols + 1):
        fig_orateurs_time.update_yaxes(range=[0, max_y], row=row, col=col)

for col in range(1, cols + 1):
    fig_orateurs_time.update_xaxes(title_text="Date", row=rows, col=col)

for row in range(1, rows + 1):
    fig_orateurs_time.update_yaxes(title_text="Nombre de mentions", row=row, col=1)

fig_orateurs_time.show()


/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_2399/1179208111.py:10: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



In [28]:
def partis_periodique(df, periode="annee", annee=2021, semaine=None, jour=None, 
                 date_debut=None, date_fin=None, jours=None, top_n=None):

    # --- Filtrage selon la période ---
    if periode == "annee":
        df_filtered = df[df["DateSeance_day"].dt.year == annee]
        titre = f"Partis mobilisant le plus la 'République' en {annee} à l'Assemblée Nationale (valeur absolue)"

    elif periode == "semaine":
        if annee is None or semaine is None:
            raise ValueError("Il faut préciser l'année et la semaine pour periode='semaine'")
        df_filtered = df[
            (df["DateSeance_day"].dt.isocalendar().year == annee) &
            (df["DateSeance_day"].dt.isocalendar().week == semaine)
        ]
        titre = f"Partis mobilisant le plus la 'République' la {semaine}e semaine {annee} à l'Assemblée Nationale (valeur absolue)"

    elif periode == "jour":
        if jour is None:
            raise ValueError("Il faut préciser la date pour periode='jour'")
        jour_dt = pd.to_datetime(jour).date()
        df_filtered = df[df["DateSeance_day"].dt.date == jour_dt]
        titre = f"Partis mobilisant le plus la 'République' le {jour_dt} (en valeur absolue)"

    elif periode == "intervalle":
        if date_debut is None or date_fin is None:
            raise ValueError("Il faut préciser date_debut et date_fin pour periode='intervalle'")
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df["DateSeance_day"] >= debut) & (df["DateSeance_day"] <= fin)]
        titre = f"Partis mobilisant le plus la 'République' du {debut.date()} au {fin.date()}"

    elif periode == "jours":
        if jours is None or not isinstance(jours, (list, tuple)):
            raise ValueError("Il faut fournir une liste de dates pour periode='jours'")
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df["DateSeance_day"].dt.date.isin(jours_dt)]
        titre = f"Partis mobilisant le plus la 'République' lors de X évènement"

    else:
        raise ValueError("periode doit être 'annee', 'semaine', 'jour', 'intervalle' ou 'jours'")

    # --- Comptage des orateurs ---
    df_counts = df_filtered["parti_affiliation"].value_counts().head(top_n).reset_index()
    df_counts.columns = ["parti_affiliation", "occurrences"]

    # --- Graphique ---
    fig = px.bar(
        df_counts,
        x="parti_affiliation",
        y="occurrences",
        title=titre,
        template="plotly_white",
    )
    fig.update_layout(xaxis_tickangle=-45, showlegend=False)
    fig.show()

In [38]:
# Partis par année
partis_periodique(df, periode="annee", annee=2024, top_n=12)

In [41]:
# Partis sur la semaine X de X
partis_periodique(df, periode="semaine", annee=2021, semaine=6)

In [ ]:
# Partis (tel jour) le X 
partis_periodique(df, periode="jour", jour="2021-02-01")


In [44]:
# Partis (sur telle période) du 1er au 16 février 2021
partis_periodique(df, periode="intervalle", date_debut="2021-02-01", date_fin="2021-02-16")

## Par personnel politique / individuellement

#### Les principaux orateurs sur la période / par législature

In [9]:
df["Nom_orateur"].value_counts()[0:50]

Nom_orateur
M. Jean-Michel Blanquer       148
M. Alexis Corbière             46
M. Gabriel Attal               29
Mme Sabine Rubin               27
M. Roger Chudeau               21
Mme Anne-Christine Lang        20
Mme Sophie Cluzel              18
Mme Michèle Victory            18
Mme Anne Brugnera              18
M. Pap Ndiaye                  18
M. Benjamin Lucas              17
M. Aurélien Pradié             17
M. Pierre-Yves Bournazel       17
Mme Sarah El Haïry             17
M. Maxime Minot                16
Mme Nicole Belloubet           15
Mme Amélie Oudéa-Castéra       14
M. Éric Ciotti                 14
Mme Béatrice Descamps          14
M. Paul Vannier                14
M. François Pupponi            13
M. Sébastien Jumel             13
M. Frédéric Reiss              12
Mme Annie Genevard             12
M. Stéphane Peu                12
M. Patrick Hetzel              12
M. Erwan Balanant              11
Mme Elsa Faucillon             11
Mme Catherine Osson            11
Mm

In [11]:
fig_top_orateurs = px.bar(
    df["Nom_orateur"].value_counts()[0:10],
    # x=top_counts.index,
    # y=top_counts.values,
    labels={"value": "Nombre d'occurence", "Personnel politique": "Orateur"},
    title="Top 20 personnel politique par occurence de la République",
    template="plotly_white",
)
fig_top_orateurs.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
)
fig_top_orateurs.show()

In [20]:
def orateurs_evolutions(df, date_col="DateSeance_day", parti_col="Nom_orateur", top_n=7):

    # Filtrer le top des partis
    top_orateurs = df[parti_col].value_counts().nlargest(top_n).index.tolist()

    # Filtrer les données pour ces partis uniquement
    df_top = df[df[parti_col].isin(top_orateurs)]

    # Grouper par année et parti
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="Y"), parti_col])
        .size()
        .reset_index(name="mentions")
    )

    # Extraire l'année pour affichage clair
    df_grouped["Année"] = df_grouped[date_col].dt.year

    # Tracer avec plotly express
    fig = px.line(
        df_grouped,
        x="Année",
        y="mentions",
        color=parti_col,
        markers=True,
        title=f"Évolution des principaux contributeur•ices aux appropriations scolaires de la République",
        labels={"mentions": "Occurrences", parti_col: "Nom"}
    )

    fig.update_layout(
        xaxis=dict(dtick=1),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Parti"
    )

    fig.show()

In [21]:
orateurs_evolutions(df)

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_2969/472274908.py:11: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



In [7]:
df_16e = df [df["legislature"]== 16]
df_15e = df [df["legislature"]== 15]

In [11]:
df_15e["nom_orateur"].value_counts()[0:20]

nom_orateur
M. Gérald Darmanin            306
M. Alexis Corbière            242
M. Jean-Luc Mélenchon         218
M. Sébastien Jumel            203
M. Jean-Michel Blanquer       180
M. Stéphane Peu               154
M. Éric Ciotti                154
M. Éric Coquerel              154
Mme Danièle Obono             133
M. Ugo Bernalicis             130
Mme Marlène Schiappa          128
M. Philippe Gosselin          126
M. Jean-Christophe Lagarde    113
Mme Mathilde Panot            112
M. Dominique Potier           102
Mme Nicole Belloubet           99
M. Pierre Dharréville          99
M. Bastien Lachaud             98
M. Edouard Philippe            88
M. Raphaël Schellenberger      87
Name: count, dtype: int64

In [54]:
df_15e["nom_orateur"].value_counts()[0:20]


nom_orateur
M. Gérald Darmanin            306
M. Alexis Corbière            242
M. Jean-Luc Mélenchon         223
M. Sébastien Jumel            203
M. Jean-Michel Blanquer       180
M. Éric Coquerel              155
M. Stéphane Peu               154
M. Éric Ciotti                154
Mme Danièle Obono             133
M. Ugo Bernalicis             130
Mme Marlène Schiappa          128
M. Philippe Gosselin          126
M. Jean-Christophe Lagarde    114
Mme Mathilde Panot            112
M. Dominique Potier           102
M. Pierre Dharréville         100
Mme Nicole Belloubet           99
M. Bastien Lachaud             98
M. Edouard Philippe            89
M. Raphaël Schellenberger      87
Name: count, dtype: int64

In [58]:
# 16e législature 
fig_top_orateurs = px.bar(
    df_16e["nom_orateur"].value_counts()[0:10],
    # x=top_counts.index,
    # y=top_counts.values,
    labels={"value": "Nombre d'occurence", "Personnel politique": "Orateur"},
    title="Top 20 personnel politique par occurence de la République lors de la 16e législature",
    template="plotly_white",
)
fig_top_orateurs.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
)
fig_top_orateurs.show()

#### Les principaux orateurs par années/semaines/séances

In [ ]:
def top_orateurs_periodique(df, periode="annee", annee=2021, semaine=None, jour=None, 
                 date_debut=None, date_fin=None, jours=None, top_n=10):

    # --- Filtrage selon la période ---
    if periode == "annee":
        df_filtered = df[df["dateSeance_day"].dt.year == annee]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' en {annee} à l'Assemblée Nationale (valeur absolue)"

    elif periode == "semaine":
        if annee is None or semaine is None:
            raise ValueError("Il faut préciser l'année et la semaine pour periode='semaine'")
        df_filtered = df[
            (df["dateSeance_day"].dt.isocalendar().year == annee) &
            (df["dateSeance_day"].dt.isocalendar().week == semaine)
        ]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' la {semaine}e semaine {annee} à l'Assemblée Nationale (valeur absolue)"

    elif periode == "jour":
        if jour is None:
            raise ValueError("Il faut préciser la date pour periode='jour'")
        jour_dt = pd.to_datetime(jour).date()
        df_filtered = df[df["dateSeance_day"].dt.date == jour_dt]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' le {jour_dt} (en valeur absolue)"

    elif periode == "intervalle":
        if date_debut is None or date_fin is None:
            raise ValueError("Il faut préciser date_debut et date_fin pour periode='intervalle'")
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df["dateSeance_day"] >= debut) & (df["dateSeance_day"] <= fin)]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' du {debut.date()} au {fin.date()}"

    elif periode == "jours":
        if jours is None or not isinstance(jours, (list, tuple)):
            raise ValueError("Il faut fournir une liste de dates pour periode='jours'")
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df["dateSeance_day"].dt.date.isin(jours_dt)]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' lors de X évènement"

    else:
        raise ValueError("periode doit être 'annee', 'semaine', 'jour', 'intervalle' ou 'jours'")

    # --- Comptage des orateurs ---
    df_counts = df_filtered["nom_orateur"].value_counts().head(top_n).reset_index()
    df_counts.columns = ["nom_orateur", "occurrences"]

    # --- Graphique ---
    fig = px.bar(
        df_counts,
        x="nom_orateur",
        y="occurrences",
        title=titre,
        template="plotly_white",
    )
    fig.update_layout(xaxis_tickangle=-45, showlegend=False)
    fig.show()

In [42]:
# Top 10 orateurs sur l'année 2021
top_orateurs_periodique(df, periode="annee", annee=2021)

In [46]:
# Top 10 orateurs sur la semaine X de X
top_orateurs_periodique(df, periode="semaine", annee=2021, semaine=6)

In [47]:
# Top 10 orateurs (tel jour) le X 
top_orateurs_periodique(df, periode="jour", jour="2021-02-01")

In [ ]:
# Top 10 orateurs (sur telle période) du 1er au 16 février 2021
top_orateurs_periodique(df, periode="intervalle", date_debut="2021-02-01", date_fin="2021-02-16")

In [52]:
# Top 10 orateurs sur des jours spécifiques
top_orateurs_periodique(df, periode="jours", jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-4", "2021-02-05"])

#### Analyse discussion du projet de loi séparatisme

In [54]:
# Top 10 orateurs lors de la 1ère discussion du projet de loi séparatisme 
top_orateurs_periodique(df, periode="jours", jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16"])

In [59]:
# Top 10 orateurs lors des discussions du projet de loi séparatisme 
top_orateurs_periodique(df, periode="jours", jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16", "2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"])



In [62]:
# Top 10 orateurs lors de la deuxième phase de discussion post débat Sénat 
top_orateurs_periodique(df, periode="jours", jours=["2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"])



### Évolution dans le temps de la fréquence d'utilisation du top X de personnels politiques

In [9]:
def stats_orateur(df, orateur, periode="semaine"):

   
    # Filtrer sur l'orateur choisi
    df_orateur = df[df["nom_orateur"] == orateur].copy()
    if df_orateur.empty:
        raise ValueError(f"Aucune donnée trouvée pour l'orateur : {orateur}")

    # Définir la granularité
    if periode == "semaine":
        df_orateur["periode"] = (
            df_orateur["dateSeance_day"].dt.isocalendar().year.astype(str)
            + "-W"
            + df_orateur["dateSeance_day"].dt.isocalendar().week.astype(str)
        )
    elif periode == "mois":
        df_orateur["periode"] = df_orateur["dateSeance_day"].dt.to_period("M").astype(str)
    elif periode == "annee":
        df_orateur["periode"] = df_orateur["dateSeance_day"].dt.year.astype(str)
    else:
        raise ValueError("periode doit être 'semaine', 'mois' ou 'annee'")

    # Compter le nombre d'interventions par période
    df_counts = df_orateur.groupby("periode").size().reset_index(name="occurrences")

    # Calculer la moyenne et la médiane
    moyenne = df_counts["occurrences"].mean()
    mediane = df_counts["occurrences"].median()

    return moyenne, mediane, df_counts


In [10]:
# Définir la personne recherchée 
orateur = "M. Gérald Darmanin"

# Statistiques par semaine
moyenne, mediane, df_semaine = stats_orateur(df, orateur, periode="semaine")
print(f"{orateur} - Moyenne/semaine: {moyenne:.2f}, Médiane/semaine: {mediane}")

# Statistiques par mois
moyenne, mediane, df_mois = stats_orateur(df, orateur, periode="mois")
print(f"{orateur} - Moyenne/mois: {moyenne:.2f}, Médiane/mois: {mediane}")

# Statistiques par année
moyenne, mediane, df_annee = stats_orateur(df, orateur, periode="annee")
print(f"{orateur} - Moyenne/an: {moyenne:.2f}, Médiane/an: {mediane}")


M. Gérald Darmanin - Moyenne/semaine: 4.62, Médiane/semaine: 2.0
M. Gérald Darmanin - Moyenne/mois: 8.41, Médiane/mois: 5.0
M. Gérald Darmanin - Moyenne/an: 58.88, Médiane/an: 37.5


In [31]:
# Graphique de l'évolution de personnes cibles par mois

# Définir la personne recherchée 
orateur = "M. Philippe Gosselin"

# Filtrer le DataFrame sur l'année choisie
df_orateur = df[df["Nom_orateur"]== orateur]

# Resampler par semaine uniquement pour cette année
df_monthly = df_orateur.resample("M", on="DateSeance_day").size().reset_index()

# Tracer le graphique
fig_time = px.bar(
    df_monthly,
    x="DateSeance_day",
    y=0,
    title=f"Évolution du nombre d'utilisation de la 'République' par Pierre-Yves Bournazel par mois",
    labels={"dateSeance_day": "Date", "Nombre de mentions": "Occurrences"},
    template="plotly_white",
)

fig_time.update_layout(showlegend=False)

fig_time.show()

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_3051/2305185215.py:10: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [103]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Top 10 orateurs
top_orateurs = df["nom_orateur"].value_counts().index[:10].tolist()

# Grouper par semaine et orateur
df_orateur = df[df["nom_orateur"].isin(top_orateurs)]
df_grouped_orateur = (
    df_orateur.groupby([pd.Grouper(key="dateSeance_day", freq="MS"), "nom_orateur"])
    .size()
    .reset_index(name="mentions")
)

cols = 4
rows = (len(top_orateurs) + cols - 1) // cols
max_y = df_grouped_orateur["mentions"].max()

fig_orateurs_time = make_subplots(
    rows=rows, cols=cols, shared_xaxes=True, subplot_titles=top_orateurs
)

for idx, orateur in enumerate(top_orateurs):
    row = idx // cols + 1
    col = idx % cols + 1
    data_orateur = df_grouped_orateur[df_grouped_orateur["nom_orateur"] == orateur]
    fig_orateurs_time.add_trace(
        go.Bar(
            x=data_orateur["dateSeance_day"], y=data_orateur["mentions"], name=orateur
        ),
        row=row,
        col=col,
    )

fig_orateurs_time.update_layout(
    height=300 * rows,
    width=1200,
    title_text="Dynamique temporelle des mentions par orateur",
    showlegend=False,
    template="plotly_white",
)

for row in range(1, rows + 1):
    for col in range(1, cols + 1):
        fig_orateurs_time.update_yaxes(range=[0, max_y], row=row, col=col)

for col in range(1, cols + 1):
    fig_orateurs_time.update_xaxes(title_text="Date", row=rows, col=col)

for row in range(1, rows + 1):
    fig_orateurs_time.update_yaxes(title_text="Nombre de mentions", row=row, col=1)

fig_orateurs_time.show()


In [ ]:
# Pour la 15e législature 

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Top 10 orateurs
top_orateurs = df_15e["nom_orateur"].value_counts().index[:4].tolist()

# Grouper par semaine et orateur
df_orateur_15e = df_15e[df_15e["nom_orateur"].isin(top_orateurs)]
df_grouped_orateur_15e = (
    df_orateur_15e.groupby([pd.Grouper(key="dateSeance_day", freq="MS"), "nom_orateur"])
    .size()
    .reset_index(name="mentions")
)

cols = 4
rows = (len(top_orateurs) + cols - 1) // cols
max_y = df_grouped_orateur_15e["mentions"].max()

fig_orateurs_time = make_subplots(
    rows=rows, cols=cols, shared_xaxes=True, subplot_titles=top_orateurs
)

for idx, orateur in enumerate(top_orateurs):
    row = idx // cols + 1
    col = idx % cols + 1
    data_orateur = df_grouped_orateur_15e[df_grouped_orateur_15e["nom_orateur"] == orateur]
    fig_orateurs_time.add_trace(
        go.Bar(
            x=data_orateur["dateSeance_day"], y=data_orateur["mentions"], name=orateur
        ),
        row=row,
        col=col,
    )

fig_orateurs_time.update_layout(
    height=300 * rows,
    width=1200,
    title_text="Dynamique temporelle des mentions par orateur",
    showlegend=False,
    template="plotly_white",
)

for row in range(1, rows + 1):
    for col in range(1, cols + 1):
        fig_orateurs_time.update_yaxes(range=[0, max_y], row=row, col=col)

for col in range(1, cols + 1):
    fig_orateurs_time.update_xaxes(title_text="Date", row=rows, col=col)

for row in range(1, rows + 1):
    fig_orateurs_time.update_yaxes(title_text="Nombre de mentions", row=row, col=1)

fig_orateurs_time.show()


## Personnel politique par groupes

In [5]:
# Fonction générale 

# Définir la personne recherchée 
parti = "UDI"

# Filtrer sur le parti choisi
df_partis = df[df["parti_affiliation"] == parti]


# Figure
fig_top_orateurs_partis = px.bar(
    df_partis["Nom_orateur"].value_counts()[0:10],
    # x=top_counts.index,
    # y=top_counts.values,
    labels={"value": "Nombre d'occurence", "Nom_orateur": "Député.es"},
    title= f"Top 10 des député.es {parti} en nb d'occurrences de la 'République'",
    template="plotly_white",
)
fig_top_orateurs_partis.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
)
fig_top_orateurs_partis.show()

# Tableaux 
df_partis["Nom_orateur"].value_counts()[0:10]

Nom_orateur
M. Jean-Christophe Lagarde    107
M. Meyer Habib                 31
M. Pascal Brindeau             29
M. Philippe Gomès              29
M. Olivier Becht               23
Mme Béatrice Descamps          18
M. Christophe Naegelen         17
M. Philippe Dunoyer            16
M. Philippe Vigier             14
Mme Maina Sage                 13
Name: count, dtype: int64

In [ ]:
# Définir la personne recherchée 
parti = "MODEM"

# Filtrer sur le parti choisi
df_partis = df[df["parti_affiliation"] == parti]    

def top_orateur_partis_periodique(df, periode="annee", annee=2021, semaine=None, jour=None, 
                 date_debut=None, date_fin=None, jours=None, top_n=10):

    # --- Filtrage selon la période ---
    if periode == "annee":
        df_filtered = df_partis[df["DateSeance_day"].dt.year == annee]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' en {annee} à l'Assemblée Nationale (valeur absolue)"

    elif periode == "semaine":
        if annee is None or semaine is None:
            raise ValueError("Il faut préciser l'année et la semaine pour periode='semaine'")
        df_filtered = df[
            (df["DateSeance_day"].dt.isocalendar().year == annee) &
            (df["DateSeance_day"].dt.isocalendar().week == semaine)
        ]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' la {semaine}e semaine {annee} à l'Assemblée Nationale (valeur absolue)"

    elif periode == "jour":
        if jour is None:
            raise ValueError("Il faut préciser la date pour periode='jour'")
        jour_dt = pd.to_datetime(jour).date()
        df_filtered = df_partis[df["DateSeance_day"].dt.date == jour_dt]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' le {jour_dt} (en valeur absolue)"

    elif periode == "intervalle":
        if date_debut is None or date_fin is None:
            raise ValueError("Il faut préciser date_debut et date_fin pour periode='intervalle'")
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df["DateSeance_day"] >= debut) & (df["DateSeance_day"] <= fin)]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' du {debut.date()} au {fin.date()}"

    elif periode == "jours":
        if jours is None or not isinstance(jours, (list, tuple)):
            raise ValueError("Il faut fournir une liste de dates pour periode='jours'")
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df_partis[df["DateSeance_day"].dt.date.isin(jours_dt)]
        titre = f"Top {top_n} du personnel politique mobilisant le plus la 'République' lors de X évènement"

    else:
        raise ValueError("periode doit être 'annee', 'semaine', 'jour', 'intervalle' ou 'jours'")

    # --- Comptage des orateurs ---
    df_counts = df_filtered["Nom_orateur"].value_counts().head(top_n).reset_index()
    df_counts.columns = ["Nom_orateur", "occurrences"]

    # --- Graphique ---
    fig = px.bar(
        df_counts,
        x="Nom_orateur",
        y="occurrences",
        title=titre,
        template="plotly_white",
    )
    fig.update_layout(xaxis_tickangle=-45, showlegend=False)
    fig.show()

In [70]:
# Top 10 orateurs sur l'année 2021
top_orateur_partis_periodique(df, periode="annee", annee=2021)

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_3051/2641435726.py:12: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



KeyError: 'nom_orateur'

#### RN

### Genre (nécessite d'avoir fait la fusion)

In [45]:
df["civ"] = df["civ"].replace({"M.": "Homme", "Mme": "Femme"})

In [46]:
fig = px.bar(df["civ"].value_counts())
fig.update_layout(
    title="Répartition des genres (civ)", template="plotly_white", showlegend=False
)
fig.show()

## Autres variables (à compléter et faire plus sérieusement)


### Mandat 

### Proximité avec la majorité 

scoreMajorite

In [6]:
df["scoreMajorite"].value_counts() 

scoreMajorite
0.000    6358
0.216     234
0.176     203
0.128     152
0.987     139
         ... 
0.931       1
0.641       1
0.868       1
0.947       1
0.328       1
Name: count, Length: 248, dtype: int64

### Expérience 

In [8]:
df["experienceDepute"].value_counts() 

experienceDepute
8 ans     2848
5 ans     1689
3 ans     1659
7 ans     1497
13 ans     553
4 ans      534
18 ans     490
2 ans      363
20 ans     274
12 ans     259
10 ans     238
15 ans     223
1 an       169
23 ans     162
14 ans     113
6 ans      112
9 ans       98
11 ans      94
22 ans      91
17 ans      76
19 ans      24
21 ans      16
4 mois      13
16 ans      13
1 mois      13
5 mois      10
6 mois       1
Name: count, dtype: int64

In [7]:
df["age"].value_counts() 

age
42.0    601
57.0    600
36.0    589
54.0    498
64.0    482
53.0    476
59.0    451
66.0    427
45.0    424
63.0    416
58.0    373
52.0    373
50.0    327
51.0    305
34.0    303
67.0    290
47.0    272
61.0    264
48.0    253
39.0    252
38.0    245
40.0    220
65.0    219
74.0    211
56.0    210
55.0    204
49.0    189
60.0    181
62.0    161
75.0    155
35.0    154
37.0    153
73.0    142
69.0    136
44.0    127
68.0    102
76.0    101
70.0     98
46.0     86
41.0     84
71.0     82
72.0     66
43.0     60
79.0     57
33.0     55
32.0     31
78.0     26
31.0     19
77.0     19
80.0     14
28.0     13
25.0     11
30.0     11
24.0      5
83.0      3
29.0      2
26.0      2
81.0      1
82.0      1
Name: count, dtype: int64

### Aire géographique 

departementCode

In [9]:
df["departementCode"].value_counts() 

departementCode
93     1023
59      918
76      537
13      502
75      491
       ... 
11        6
43        6
19        5
15        1
986       1
Name: count, Length: 107, dtype: int64

## Tests régressions linéaires

In [7]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

def regression_lineaire(y_col, x_cols):
    """
    Effectue une régression linéaire simple ou multiple à partir d'un CSV.
    
    csv_path : chemin du fichier CSV
    y_col : nom de la colonne cible (str)
    x_cols : liste des colonnes explicatives (list de str)
    """
   
    
    # Définir X et y
    X = df[x_cols].values
    y = df[y_col].values
    
    # Créer et entraîner le modèle
    model = LinearRegression()
    model.fit(X, y)
    
    # Prédictions
    y_pred = model.predict(X)
    
    # Résultats
    print("✅ Régression terminée")
    print("Coefficient(s):", model.coef_)
    print("Intercept:", model.intercept_)
    print("Score R²:", model.score(X, y))
    
    # Si une seule variable explicative, on affiche un graphique
    if len(x_cols) == 1:
        plt.scatter(X, y, color="blue", label="Données réelles")
        plt.plot(X, y_pred, color="red", label="Régression")
        plt.xlabel(x_cols[0])
        plt.ylabel(y_col)
        plt.legend()
        plt.show()
    
    return model


In [8]:
modele = regression_lineaire("../data/interim/df_repu.csv", y_col=["Texte"], x_cols=["parti_affiliation", "Nom_orateur", "age"])

TypeError: regression_lineaire() got multiple values for argument 'y_col'

In [ ]:
df